# SQ-3 — Composability rule battery (against hand-authored expectations)

**Tests:** do the FLTA composability rules fire on mis-configured chains and stay silent on well-configured ones, *against a manifest authored independently from the rule engine*?

**Why this matters.** Earlier iterations of this notebook computed the expected firings from the same `chains.py` logic that drives `rules.py`. That is the rule engine agreeing with itself. The expectations now live in [`card/battery_expected.json`](../../card/battery_expected.json) — hand-authored by reading each chain's `pet_components`, ε, and jurisdictional context and applying the rule definitions from first principles.

**Headline:** per-rule TPR (rule fires when expected) and TNR (rule silent when expected); both 100% on this battery.

**Threshold provenance.** `COMP-GRADINV-001` triggers at ε > 4.0, pinned to Hatamizadeh *et al.* (CVPR 2023) §5 and Boenisch *et al.* (USENIX Security 2023) §6 — see [`flta_eval/rules.py`](../../flta_eval/rules.py).

In [1]:
import json, sys, time
from pathlib import Path

EVAL_DIR = Path.cwd().parents[1] if Path.cwd().name.startswith('sq') else Path.cwd()
sys.path.insert(0, str(EVAL_DIR))

from flta_eval import audit, chains, rules
battery = chains.build_battery()
expected_doc = json.loads((EVAL_DIR / 'card' / 'battery_expected.json').read_text())
expected = {e['chain_id']: set(e['expected_firings']) for e in expected_doc['chains']}
print(f'battery size: {len(battery)}; hand-authored expectations: {len(expected)}')

battery size: 16; hand-authored expectations: 16


In [2]:
RULES = ['COMP-SECAGG-001', 'COMP-GRADINV-001', 'COMP-TEEMPC-001', 'COMP-JURIS-001']

t0 = time.time()
rows = []
rule_outcomes = {r: {'TP': 0, 'FP': 0, 'TN': 0, 'FN': 0} for r in RULES}
for entry in battery:
    report = rules.evaluate_card(entry['card'])
    fired = {f['rule_id'] for f in report['findings'] if f['rule_id'].startswith('COMP-')}
    exp = expected.get(entry['chain_id'], set())
    rows.append((entry['chain_id'], sorted(exp), sorted(fired), 'OK' if fired == exp else 'MISMATCH'))
    for r in RULES:
        if r in exp and r in fired:    rule_outcomes[r]['TP'] += 1
        elif r in exp:                  rule_outcomes[r]['FN'] += 1
        elif r in fired:                rule_outcomes[r]['FP'] += 1
        else:                            rule_outcomes[r]['TN'] += 1
elapsed = time.time() - t0
print(f'elapsed: {elapsed:.3f}s')
print()
print(f"{'chain_id':45s} {'status':9s} expected → fired")
for cid, exp, fired, status in rows:
    es = ','.join(e.removeprefix('COMP-') for e in exp) or '—'
    fs = ','.join(f.removeprefix('COMP-') for f in fired) or '—'
    print(f'  {cid:43s} {status:8s}  {es:25s} → {fs}')

elapsed: 0.001s

chain_id                                      status    expected → fired
  FL__H1_default                              OK        SECAGG-001                → SECAGG-001
  FL__H2_gradinv_eps6                         OK        SECAGG-001                → SECAGG-001
  FL__H3_juris_unmechanised                   OK        JURIS-001,SECAGG-001      → JURIS-001,SECAGG-001
  FL__H4_well_configured                      OK        —                         → —
  FL+DP__H1_default                           OK        GRADINV-001,SECAGG-001    → GRADINV-001,SECAGG-001
  FL+DP__H2_gradinv_eps6                      OK        GRADINV-001,SECAGG-001    → GRADINV-001,SECAGG-001
  FL+DP__H3_juris_unmechanised                OK        GRADINV-001,JURIS-001,SECAGG-001 → GRADINV-001,JURIS-001,SECAGG-001
  FL+DP__H4_well_configured                   OK        —                         → —
  FL+TEE+DP__H1_default                       OK        GRADINV-001,SECAGG-001,TEEMPC-001 → GRADINV-001,S

In [3]:
print(f"{'rule':22s} {'TP':>4s} {'FN':>4s} {'FP':>4s} {'TN':>4s} {'TPR':>7s} {'TNR':>7s}")
for r in RULES:
    o = rule_outcomes[r]
    tpr = o['TP'] / (o['TP'] + o['FN']) if (o['TP'] + o['FN']) else 1.0
    tnr = o['TN'] / (o['TN'] + o['FP']) if (o['TN'] + o['FP']) else 1.0
    print(f"  {r:20s} {o['TP']:>4d} {o['FN']:>4d} {o['FP']:>4d} {o['TN']:>4d} {tpr:>6.2%} {tnr:>6.2%}")

rule                     TP   FN   FP   TN     TPR     TNR
  COMP-SECAGG-001         9    0    0    7 100.00% 100.00%
  COMP-GRADINV-001        6    0    0   10 100.00% 100.00%
  COMP-TEEMPC-001         3    0    0   13 100.00% 100.00%
  COMP-JURIS-001          4    0    0   12 100.00% 100.00%


In [4]:
rule_metrics = {r: {'tp': o['TP'], 'fn': o['FN'], 'fp': o['FP'], 'tn': o['TN'],
                    'tpr': o['TP']/(o['TP']+o['FN']) if (o['TP']+o['FN']) else 1.0,
                    'tnr': o['TN']/(o['TN']+o['FP']) if (o['TN']+o['FP']) else 1.0}
                for r, o in rule_outcomes.items()}

record_path = audit.write_result_record(
    target_dir=EVAL_DIR / 'results' / 'sq3',
    attack='rule_battery',
    variant='SQ-3',
    threat_profile='I',
    dataset={'name': 'battery_16_chains_v2', 'n': len(battery),
             'expectations_source': 'card/battery_expected.json (hand-authored)'},
    config={'rules_tested': RULES,
            'gradinv_threshold_epsilon': rules.GRADINV_EPSILON_THRESHOLD},
    seed_namespace='sq3.rule_battery.v2',
    result={
        'rule_metrics': rule_metrics,
        'mismatches': [{'chain_id': cid, 'expected': exp, 'fired': fired}
                       for cid, exp, fired, s in rows if s != 'OK'],
    },
    tolerances={'tpr_per_rule': 1.0, 'tnr_per_rule': 1.0},
)
print(f'wrote {record_path.relative_to(EVAL_DIR)}')

wrote results/sq3/rule_battery__SQ-3__2026-05-26T00-31-35Z.json
